# Predict Ahlqvist classification
Analyais of pre-COVID19 UAB data by Ahlqvist classification showed racial differences in cluster distribution. Map cluster assignments from pre-COVID19 data to post-COVID19 data.

**Objective:** Construct a machine learning model to predict cluster assignment.

# Environment setup
Import required modules.

In [50]:
import numpy as np
import pandas as pd
#import seaborn as sns
#import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report

#from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

#from sklearn.inspection import permutation_importance


Set random seed for reproducibility.

In [2]:
rng = np.random.RandomState(42)

Import cleaned data.

In [3]:
data_path = "/data/user/brianlu/projects/precision_diabetes/Clustering/exports/manuscript_Ahlqvist_analysis_2024-05-03/outpatient.csv"
data = pd.read_csv(data_path)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Columns: 300 entries, DE_ID to aa_factor
dtypes: float64(93), int64(41), object(166)
memory usage: 2.7+ MB


Set up train-test split.

In [4]:
predictors = ["gad", "a1c", "bmi", "diabetes_age", "homa_cpeptide_b", "homa_cpeptide_ir"]

# multiple classes to predict, transform into labels
le = LabelEncoder()
x = data[predictors]
le.fit(data["Ahlqvist"])
le.classes_ = np.array(["SAID", "SIDD", "SIRD", "MOD", "MARD"])
y_ahlqvist = le.transform(data["Ahlqvist"])

x_train, x_test, y_train, y_test = train_test_split(x, y_ahlqvist, test_size = 0.2, shuffle = True, random_state = rng)

def extract_scores(model):
    # find index of best model
    index = np.where(model.cv_results_["rank_test_f1_macro"] == 1)[0][0]
    
    # extract scores of best model
    accuracy = model.cv_results_["mean_test_accuracy"][index]
    precision = model.cv_results_["mean_test_precision_macro"][index]
    recall = model.cv_results_["mean_test_recall_macro"][index]
    f1 = model.cv_results_["mean_test_f1_macro"][index]
    roc = model.cv_results_["mean_test_roc_auc_ovr"][index]
    
    out = pd.DataFrame({"metric" : ["accuracy", "precision", "recall", "f1", "roc"], 
                        "scores" : [accuracy, precision, recall, f1, roc]})
    
    return out

# Model selection

Best model parameters were selected by GridSearch with 5-fold cross validation repeated 5 times refit using weighted F1 scores.

In [42]:
cv = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = rng)
refit = "f1_weighted"

Weighted scores were used to accoung for class imbalance. Scores for each class were also inspected for each model.

In [6]:
score = ["balanced_accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc_ovr_weighted"]

Function to extract scoring metrics.

In [7]:
def extract_scores(model):
    # find index of best model
    index = np.where(model.cv_results_["rank_test_f1_weighted"] == 1)[0][0]
    
    # extract scores of best model
    accuracy = model.cv_results_["mean_test_balanced_accuracy"][index]
    precision = model.cv_results_["mean_test_precision_weighted"][index]
    recall = model.cv_results_["mean_test_recall_weighted"][index]
    f1 = model.cv_results_["mean_test_f1_weighted"][index]
    roc = model.cv_results_["mean_test_roc_auc_ovr_weighted"][index]
    
    out = pd.DataFrame({"metric" : ["accuracy", "precision", "recall", "f1", "roc"], 
                        "scores" : [accuracy, precision, recall, f1, roc]})
    
    return out

def extract_scores(model):
    # find index of best model
    index = np.where(model.cv_results_["rank_test_f1_macro"] == 1)[0][0]
    
    # extract scores of best model
    accuracy = model.cv_results_["mean_test_accuracy"][index]
    precision = model.cv_results_["mean_test_precision_macro"][index]
    recall = model.cv_results_["mean_test_recall_macro"][index]
    f1 = model.cv_results_["mean_test_f1_macro"][index]
    roc = model.cv_results_["mean_test_roc_auc_ovr"][index]
    
    out = pd.DataFrame({"metric" : ["accuracy", "precision", "recall", "f1", "roc"], 
                        "scores" : [accuracy, precision, recall, f1, roc]})
    
    return out

## Logistic regression

In [8]:
%%time

from sklearn.linear_model import LogisticRegression

parameters = {"C" : [0.1, 1, 2, 2.5, 5, 10, 20, 40, 100, 200]}

clf = LogisticRegression(random_state = rng, max_iter = 10000, n_jobs = -1)

lr_gs = GridSearchCV(estimator = clf, 
                     param_grid = parameters,
                     cv = cv,
                     scoring = score,
                     refit = refit)

lr_gs.fit(x_train, y_train)

# save score for comparision
lr_score = extract_scores(lr_gs)

{'C': 40}
LogisticRegression(C=40, max_iter=10000, n_jobs=-1,
                   random_state=RandomState(MT19937) at 0x2AAAFA283740)
CPU times: user 3.1 s, sys: 121 ms, total: 3.23 s
Wall time: 4min 2s


In [141]:
print(lr_gs.best_params_)
print(lr_gs.best_estimator_)

{'C': 40}
LogisticRegression(C=40, max_iter=10000, n_jobs=-1,
                   random_state=RandomState(MT19937) at 0x2AAAFA283740)


In [111]:
print(classification_report(y_train, 
                      lr_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       0.87      0.88      0.87       235
           2       0.86      0.84      0.85       133
           3       0.83      0.82      0.83       300
           4       0.85      0.86      0.86       205

    accuracy                           0.86       955
   macro avg       0.88      0.88      0.88       955
weighted avg       0.86      0.86      0.86       955



## SGD

In [9]:
%time

from sklearn.linear_model import SGDClassifier

parameters = {"loss" : ["log_loss", "modified_huber"],
              "penalty" : ["l2", "l1", "elasticnet"],
              "alpha" : [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]}

clf = SGDClassifier(random_state = rng, n_jobs = -1)

sgd_gs = GridSearchCV(estimator = clf, 
                     param_grid = parameters,
                     cv = cv,
                     scoring = score,
                     refit = refit)

sgd_gs.fit(x_train, y_train)

# save score for comparision
sgd_score = extract_scores(sgd_gs)

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 4.29 µs


/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/linear_model/_base.py:472: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 401, in _score
    return self._sign * self._score_func(y, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/p

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/linear_model/_base.py:472: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 401, in _score
    return self._sign * self._score_func(y, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/p

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/linear_model/_base.py:472: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 401, in _score
    return self._sign * self._score_func(y, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/p

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/linear_model/_base.py:472: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 401, in _score
    return self._sign * self._score_func(y, y_pred, **scoring_kwargs)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/p

...

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:713: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:713: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: 

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:713: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:713: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: 

{'alpha': 0.001, 'loss': 'modified_huber', 'penalty': 'l1'}
SGDClassifier(alpha=0.001, loss='modified_huber', n_jobs=-1, penalty='l1',
              random_state=RandomState(MT19937) at 0x2AAAFA73D440)


/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_search.py:976: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan 0.82496333 0.88213985 0.83029808
 0.88818316 0.90141639 0.90446966 0.81069312 0.89390887 0.822033
 0.89845839 0.90450659 0.9083766  0.80189081 0.89973609 0.84917654
 0.91495899 0.87518527 0.91846864 0.82923574 0.88742522 0.84750764
 0.8944183  0.7385169  0.88139116 0.89412121 0.80338047 0.88519697
 0.84472974 0.56293876 0.68783898 0.89184321 0.59537559 0.8143229
 0.77556446 0.5        0.552889   0.83680665 0.5        0.51810618]
  warnings.warn(


In [120]:
print(sgd_gs.best_params_)
print(sgd_gs.best_estimator_)

{'alpha': 0.001, 'loss': 'modified_huber', 'penalty': 'l1'}
SGDClassifier(alpha=0.001, loss='modified_huber', n_jobs=-1, penalty='l1',
              random_state=RandomState(MT19937) at 0x2AAAFA73D440)


In [112]:
print(classification_report(y_train, 
                      sgd_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       0.40      0.51      0.45        82
           1       0.71      0.92      0.80       235
           2       0.71      0.75      0.73       133
           3       0.86      0.62      0.72       300
           4       0.82      0.75      0.79       205

    accuracy                           0.73       955
   macro avg       0.70      0.71      0.70       955
weighted avg       0.75      0.73      0.73       955



## SVM

In [10]:
%%time

from sklearn.svm import LinearSVC

parameters = {"penalty" : ["l1", "l2"],
              "C" : [0.001, 0.01, 0.1, 1, 10, 100]}

clf = LinearSVC(dual = "auto", random_state = rng, max_iter = 10000)

svc_gs = GridSearchCV(estimator = clf, 
                     param_grid = parameters,
                     cv = cv,
                     scoring = score,
                     refit = refit)

svc_gs.fit(x_train, y_train)

# save score for comparision
svc_score = extract_scores(svc_gs)

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 86, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/

...

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 86, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 86, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 86, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/

{'C': 10, 'penalty': 'l2'}
LinearSVC(C=10, dual='auto', max_iter=10000,
          random_state=RandomState(MT19937) at 0x2AAAFA73DC40)
CPU times: user 6.54 s, sys: 8.7 ms, total: 6.55 s
Wall time: 6.71 s


/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:842: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 136, in __call__
    score = scorer._score(
            ^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 399, in _score
    y_pred = method_caller(clf, "predict_proba", X, pos_label=self._get_pos_label())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 86, in _cached_call
    result, _ = _get_response_values(
                ^^^^^^^^^^^^^^^^^^^^^
  File "/share/apps/rc/software/

In [142]:
print(svc_gs.best_params_)
print(svc_gs.best_estimator_)

{'C': 10, 'penalty': 'l2'}
LinearSVC(C=10, dual='auto', max_iter=10000,
          random_state=RandomState(MT19937) at 0x2AAAFA73DC40)


In [113]:
print(classification_report(y_train, 
                      svc_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       0.84      0.87      0.85       235
           2       0.88      0.81      0.84       133
           3       0.82      0.79      0.81       300
           4       0.83      0.88      0.86       205

    accuracy                           0.85       955
   macro avg       0.87      0.87      0.87       955
weighted avg       0.85      0.85      0.85       955



## K-nearest neighbor

In [11]:
%%time 

from sklearn.neighbors import KNeighborsClassifier

knn_clf = KNeighborsClassifier(n_jobs = -1)

parameters = {'n_neighbors': range(5, 36, 10), 
             'weights': ['uniform', 'distance'],
             'leaf_size':[1, 3, 5, 10],
             'algorithm':['auto', 'kd_tree']}

knn_gs = GridSearchCV(estimator = knn_clf,
                    param_grid = parameters,
                    cv = cv,
                    scoring = score,
                    refit = refit)

knn_gs.fit(x_train, y_train)

# save score for comparision
knn_score = extract_scores(knn_gs)

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

...

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

{'algorithm': 'auto', 'leaf_size': 1, 'n_neighbors': 5, 'weights': 'distance'}
KNeighborsClassifier(leaf_size=1, n_jobs=-1, weights='distance')
CPU times: user 27.1 s, sys: 904 ms, total: 28 s
Wall time: 34.9 s


/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

In [121]:
print(knn_gs.best_params_)
print(knn_gs.best_estimator_)

{'algorithm': 'auto', 'leaf_size': 1, 'n_neighbors': 5, 'weights': 'distance'}
KNeighborsClassifier(leaf_size=1, n_jobs=-1, weights='distance')


In [114]:
print(classification_report(y_train, 
                      knn_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.00      1.00       235
           2       1.00      1.00      1.00       133
           3       1.00      1.00      1.00       300
           4       1.00      1.00      1.00       205

    accuracy                           1.00       955
   macro avg       1.00      1.00      1.00       955
weighted avg       1.00      1.00      1.00       955



## Decision tree

In [12]:
%%time

from sklearn import tree

parameters = {"criterion": ["gini", "entropy", "log_loss"],
              'min_samples_leaf': range(1, 21, 1),
              "max_depth" : [5, 10, 15, 20, None]}

tree_clf = tree.DecisionTreeClassifier(random_state = rng)

tree_gs = GridSearchCV(estimator = tree_clf, 
                     param_grid = parameters,
                     cv = cv,
                     scoring = score,
                     refit = refit)

tree_gs.fit(x_train, y_train)

# save score for comparision
tree_score = extract_scores(tree_gs)

/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/share/apps/rc/software/Anaconda3/2023.07-2/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/sha

{'criterion': 'entropy', 'max_depth': 15, 'min_samples_leaf': 1}
DecisionTreeClassifier(criterion='entropy', max_depth=15,
                       random_state=RandomState(MT19937) at 0x2AAAFA73D540)
CPU times: user 1min 39s, sys: 41.3 ms, total: 1min 39s
Wall time: 1min 40s


In [122]:
print(tree_gs.best_params_)
print(tree_gs.best_estimator_)

{'criterion': 'entropy', 'max_depth': 15, 'min_samples_leaf': 1}
DecisionTreeClassifier(criterion='entropy', max_depth=15,
                       random_state=RandomState(MT19937) at 0x2AAAFA73D540)


In [115]:
print(classification_report(y_train, 
                      tree_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.00      1.00       235
           2       1.00      1.00      1.00       133
           3       1.00      1.00      1.00       300
           4       1.00      1.00      1.00       205

    accuracy                           1.00       955
   macro avg       1.00      1.00      1.00       955
weighted avg       1.00      1.00      1.00       955



## Random forest

In [ ]:
%%time

from sklearn.ensemble import RandomForestClassifier

parameters = {'min_samples_leaf': range(2, 8, 2),
              "max_depth" : [5, 10, 15, 20, 30, None],
              "max_features" : [1, 2, 3],
              "n_estimators" : [50, 100, 200, 300, 400]}

rf_clf = RandomForestClassifier(random_state = rng, n_jobs = -1)
rf_gs = GridSearchCV(estimator = rf_clf, 
                 param_grid = parameters,
                 cv = cv,
                 scoring = score,
                 refit = refit)

rf_gs.fit(x_train, y_train)

# save score for comparision
rf_score = extract_scores(rf_gs)

In [123]:
print(rf_gs.best_params_)
print(rf_gs.best_estimator_)

{'max_depth': 15, 'max_features': 2, 'min_samples_leaf': 2, 'n_estimators': 200}
RandomForestClassifier(max_depth=15, max_features=2, min_samples_leaf=2,
                       n_estimators=200, n_jobs=-1,
                       random_state=RandomState(MT19937) at 0x2AAAFA73DA40)


In [116]:
print(classification_report(y_train, 
                      rf_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      0.99      1.00       235
           2       1.00      0.99      1.00       133
           3       0.99      1.00      0.99       300
           4       1.00      0.99      0.99       205

    accuracy                           0.99       955
   macro avg       1.00      0.99      1.00       955
weighted avg       0.99      0.99      0.99       955



## Gradient boosting

In [ ]:
%%time

from sklearn.ensemble import GradientBoostingClassifier

parameters = {"loss" : ["log_loss"],
              "learning_rate" : [0.01, 0.05, 0.1, 0.2, 0.3],
              "n_estimators" : [50, 100, 200],
              "criterion" : ["friedman_mse", "squared_error"],
              "max_depth" : range(2, 7)}

gb_clf = GradientBoostingClassifier(random_state = rng)

gb_gs = GridSearchCV(estimator = gb_clf, 
                 param_grid = parameters,
                 cv = cv,
                 scoring = score,
                 refit = refit)

gb_gs.fit(x_train, y_train)

# save score for comparision
gb_score = extract_scores(gb_gs)

In [124]:
print(gb_gs.best_params_)
print(gb_gs.best_estimator_)

{'criterion': 'friedman_mse', 'learning_rate': 0.1, 'loss': 'log_loss', 'max_depth': 6, 'n_estimators': 200}
GradientBoostingClassifier(max_depth=6, n_estimators=200,
                           random_state=RandomState(MT19937) at 0x2AAAFA73D140)


In [117]:
print(classification_report(y_train, 
                      gb_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.00      1.00       235
           2       1.00      1.00      1.00       133
           3       1.00      1.00      1.00       300
           4       1.00      1.00      1.00       205

    accuracy                           1.00       955
   macro avg       1.00      1.00      1.00       955
weighted avg       1.00      1.00      1.00       955



## Histogram-based gradient boosting

In [ ]:
%%time

from sklearn.ensemble import HistGradientBoostingClassifier

parameters = {"learning_rate" : [0.01, 0.05, 0.1, 0.2, 0.3, 1],
              "min_samples_leaf" : range(2, 22, 2),
              "categorical_features" : [["gad"], None],
              "max_depth" : [5, 10, 20, None],
              "interaction_cst"  : ["pairwise", "no_interactions", None]}

hgb_clf = HistGradientBoostingClassifier(random_state = rng)

hgb_gs = GridSearchCV(estimator = hgb_clf, 
                 param_grid = parameters,
                 cv = cv,
                 scoring = score,
                 refit = refit)

hgb_gs.fit(x_train, y_train)

# save score for comparision
hgb_score = extract_scores(hgb_gs)

In [125]:
print(hgb_gs.best_params_)
print(hgb_gs.best_estimator_)

{'categorical_features': ['gad'], 'interaction_cst': None, 'learning_rate': 0.3, 'max_depth': 20, 'min_samples_leaf': 8}
HistGradientBoostingClassifier(categorical_features=['gad'], learning_rate=0.3,
                               max_depth=20, min_samples_leaf=8,
                               random_state=RandomState(MT19937) at 0x2AAAFA73E240)


In [118]:
print(classification_report(y_train, 
                      hgb_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.00      1.00       235
           2       1.00      1.00      1.00       133
           3       1.00      1.00      1.00       300
           4       1.00      1.00      1.00       205

    accuracy                           1.00       955
   macro avg       1.00      1.00      1.00       955
weighted avg       1.00      1.00      1.00       955



## AdaBoost

In [ ]:
%%time

from sklearn.ensemble import AdaBoostClassifier

parameters = {"learning_rate" : [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1, 2],
              "n_estimators" : [50, 100, 250, 400, 500]}

ada_clf = AdaBoostClassifier(random_state = rng)

ada_gs = GridSearchCV(estimator = ada_clf, 
                 param_grid = parameters,
                 cv = cv,
                 scoring = score,
                 refit = refit)

ada_gs.fit(x_train, y_train)

# save score for comparision
ada_score = extract_scores(ada_gs)

In [126]:
print(ada_gs.best_params_)
print(ada_gs.best_estimator_)

{'learning_rate': 0.5, 'n_estimators': 50}
AdaBoostClassifier(learning_rate=0.5,
                   random_state=RandomState(MT19937) at 0x2AAAFA283C40)


In [119]:
print(classification_report(y_train, 
                      ada_gs.best_estimator_.predict(x_train),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       0.88      0.89      0.88       235
           2       0.92      0.68      0.78       133
           3       0.78      0.82      0.80       300
           4       0.80      0.87      0.83       205

    accuracy                           0.84       955
   macro avg       0.88      0.85      0.86       955
weighted avg       0.85      0.84      0.84       955



## Model scores

In [23]:
pd.DataFrame({"metric": lr_score["metric"],
              "logistic_regression" : lr_score["scores"],
              "sgd" : sgd_score["scores"],
              "svm" : svc_score["scores"],
              "knn" : knn_score["scores"],
              "decision_tree" : tree_score['scores'],
              "random_forest" : rf_score["scores"],
              "gradient_boost" : gb_score["scores"],
              "hist_gradient_boost" : hgb_score["scores"],
              "ada_boost" : ada_score["scores"]})

,metric,logistic_regression,sgd,svm,knn,decision_tree,random_forest,gradient_boost,hist_gradient_boost,ada_boost
0,accuracy,0.870584,0.669103,0.866272,0.628068,0.894044,0.907670,0.918003,0.924109,0.825291
1,precision,0.858259,0.753430,0.849813,0.678011,0.881124,0.897019,0.912095,0.919747,0.823172
2,recall,0.855916,0.692984,0.845445,0.700942,0.878743,0.894450,0.909110,0.918325,0.814450
3,f1,0.855844,0.706462,0.845287,0.677714,0.878714,0.894487,0.909366,0.918194,0.813082
4,roc,0.969773,0.893909,NaN,0.870582,0.919262,0.987863,0.987885,0.991483,0.943160


In [28]:
#multiclass


plt.plot(
    fpr["micro"],
    tpr["micro"],
    label=f"micro-average ROC curve (AUC = {roc_auc['micro']:.2f})",
    color="deeppink",
    linestyle=":",
    linewidth=4,
)

plt.plot(
    fpr["macro"],
    tpr["macro"],
    label=f"macro-average ROC curve (AUC = {roc_auc['macro']:.2f})",
    color="navy",
    linestyle=":",
    linewidth=4,
)

Check performance for each class.

np.select(conditions, choices) for case_when equivalent on ovr-optimized threshold with TunedThresholdClassifierCV. may need a mixed class.

from itertools import cycle
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import RocCurveDisplay
fig, ax = plt.subplots(figsize=(6, 6))

label_binarizer = LabelBinarizer().fit(y_train)
y_onehot_test = label_binarizer.transform(y_test)
y_score = hgb_gs.best_estimator_.predict_proba(x_test)

colors = cycle(["plum", "lightsalmon","cornflowerblue", "powderblue", "darkseagreen"])
#colors = cycle(["powderblue", "darkseagreen", "lightsalmon", "plum"])
target_names = ["SAID", "SIDD", "SIRD", "MOD", "MARD"]

for class_id, color in zip(range(len(target_names)), colors):
    RocCurveDisplay.from_predictions(
        y_onehot_test[:, class_id],
        y_score[:, class_id],
        name=f"ROC curve for {target_names[class_id]}",
        color=color,
        ax=ax,
        plot_chance_level=(class_id == 2),
    )

_ = ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="Extension of Receiver Operating Characteristic\nto One-vs-Rest multiclass",
)

from sklearn.metrics import precision_recall_curve

precision = dict()
recall = dict()
for class_id, color in zip(range(len(target_names)), colors):
    #RocCurveDisplay.from_predictions(
    #    y_onehot_test[:, class_id],
    #    y_score[:, class_id],
    #    name=f"ROC curve for {target_names[class_id]}",
    #    color=color,
    #    ax=ax,
    #    plot_chance_level=(class_id == 2),
    #)
    
    #PrecisionRecallDisplay.from_estimator(rf, x_test, y_onehot_test[:, class_id], plot_chance_level=True, ax = ax)
    precision[class_id], recall[class_id], _ = precision_recall_curve(y_onehot_test[:, class_id],
                                                        y_score[:, class_id])
    plt.plot(recall[class_id], precision[class_id], lw=2, label='class {}'.format(class_id))
    
#_ = ax.set(
#    xlabel="False Positive Rate",
##    ylabel="True Positive Rate",
#    title="Extension of Receiver Operating Characteristic\nto One-vs-Rest multiclass",
#)

plt.ylabel("precision")
plt.legend(loc="best")

#prc_disp = PrecisionRecallDisplay.from_estimator(final_model, x_test, y_test, plot_chance_level=True, ax = ax)


from sklearn.metrics import RocCurveDisplay
ax = plt.gca()
rfc_disp = RocCurveDisplay.from_estimator(rf_gs.best_estimator_, x_test, y_test, ax=ax, alpha=0.8, plot_chance_level=True)
plt.show()

from sklearn.metrics import PrecisionRecallDisplay
PrecisionRecallDisplay.from_estimator(rf_gs.best_estimator_, x_test, y_test, plot_chance_level=True)

importance = rf_gs.best_estimator_.feature_importances_
std = np.std([rf_gs.best_estimator_.feature_importances_ for tree in rf_gs.best_estimator_.estimators_], axis=0)

rf_importances = pd.Series(importance, index = x_train.columns.tolist())

fig, ax = plt.subplots()
rf_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

rf_importances

# Ensemble meta-models

## Voting classifier
Select models with top 3 F1 scores and all models to vote. Decision tree was selected over gradient boosting since histogram-based gradient boosting is an extension of gradient boosting already. SVM was exluded since SVM doesn't have the `predict_proba` method.

In [130]:
from sklearn.ensemble import VotingClassifier

vote_3 = VotingClassifier(estimators = [#("lr", lr_gs.best_estimator_),
                                      #("sgd", sgd_gs.best_estimator_),
                                      #("tree", tree_gs.best_estimator_),
                                      ("rf", rf_gs.best_estimator_),
                                      ("gb", gb_gs.best_estimator_),
                                      ("hgb", gb_gs.best_estimator_)#,
                                      #("ada", ada_gs.best_estimator_)
                                          ],
                        voting = "soft",
                         n_jobs = -1)

vote_all = VotingClassifier(estimators = [("lr", lr_gs.best_estimator_),
                                      ("sgd", sgd_gs.best_estimator_),
                                      ("tree", tree_gs.best_estimator_),
                                      ("rf", rf_gs.best_estimator_),
                                      ("gb", gb_gs.best_estimator_),
                                      ("hgb", gb_gs.best_estimator_),
                                      ("ada", ada_gs.best_estimator_)
],
                        voting = "soft",
                           n_jobs = -1)

## Stacking classifier
Select models with top 3 F1 scores and all models to stack. Decision tree was selected over gradient boosting since histogram-based gradient boosting is an extension of gradient boosting already. SVM was exluded since SVM doesn't have the `predict_proba` method. Stacking classifiers only works with stratified k fold cross validation.

In [144]:
from sklearn.ensemble import StackingClassifier

secv = StratifiedKFold(n_splits=5, shuffle = True, random_state = rng)

stack_3 = StackingClassifier(estimators = [#("lr", lr_gs.best_estimator_),
                                      #("sgd", sgd_gs.best_estimator_),
                                      #("tree", tree_gs.best_estimator_),
                                      ("rf", rf_gs.best_estimator_),
                                      ("gb", gb_gs.best_estimator_),
                                      ("hgb", gb_gs.best_estimator_)#,
                                      #("ada", ada_gs.best_estimator_)
                                          ], cv = secv, n_jobs = -1)

stack_all = StackingClassifier(estimators = [("lr", lr_gs.best_estimator_),
                                      ("sgd", sgd_gs.best_estimator_),
                                      ("tree", tree_gs.best_estimator_),
                                      ("rf", rf_gs.best_estimator_),
                                      ("gb", gb_gs.best_estimator_),
                                      ("hgb", gb_gs.best_estimator_),
                                      ("ada", ada_gs.best_estimator_)
                                          ], cv = secv, n_jobs = -1)


## Model comparison
Compare average F1 scores of individual models to ensemble models after 5-fold cross validation repeated 5 times to determine score stability.

In [145]:
# declare k-fold for model comparison
mc_ecv = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = rng)

In [146]:
%%time

# Create a new list of labels that includes Voting Classifiers
labels_new = ['lr', 'sgd', "svm", 'tree', "rf", "gb", "hgd", "ada", 
              "vote_top3", "vote_all", "stacking_top3", "stacking_all"]

models = [lr_gs.best_estimator_, 
          sgd_gs.best_estimator_,
          svc_gs.best_estimator_,
          tree_gs.best_estimator_,
          rf_gs.best_estimator_,
          gb_gs.best_estimator_,
          hgb_gs.best_estimator_,
          ada_gs.best_estimator_,
          vote_3, 
          vote_all,
          stack_3, 
          stack_all]        

# declare list to collect scores
model_scores = []
# Loop through the classifiers and perform 5-fold cross-validation for each
for clf, label in zip(models, labels_new):
    
    # Use cross_val_score to compute accuracy scores using 5-fold cross-validation
    scores = cross_val_score(clf, x_train, y_train, cv = mc_ecv, scoring = 'f1_weighted', n_jobs = -1)
     
    # store scores in df
    model_scores.append(pd.DataFrame({"model" : [label],
                                      "mean f1" : [scores.mean()],
                                      "f1 std" : [scores.std()],
                                      "median f1" : [np.median(scores)]}))
# print scores    
pd.concat(model_scores)

CPU times: user 742 ms, sys: 67.4 ms, total: 809 ms
Wall time: 10min 14s


,model,mean f1,f1 std,median f1
0,lr,0.855060,0.025510,0.862512
0,sgd,0.693497,0.042757,0.700262
0,svm,0.843654,0.018016,0.843126
0,tree,0.877984,0.020185,0.879656
0,rf,0.889989,0.021530,0.890017
0,gb,0.899163,0.016599,0.900063
0,hgd,0.916862,0.020917,0.915757
0,ada,0.816707,0.043042,0.831135
0,vote_top3,0.902416,0.021038,0.900088
0,vote_all,0.905252,0.019005,0.905797


# Final model
Test final model on test data.

In [99]:
final_model = hgb_gs.best_estimator_
y_pred = final_model.predict(x_test)

In [110]:
pd.DataFrame({"metric" : ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc"],
              "score" : [accuracy_score(y_test, y_pred),
                         balanced_accuracy_score(y_test, y_pred),
                         precision_score(y_test, y_pred, average = "weighted"),
                         recall_score(y_test, y_pred, average = "weighted"),
                         f1_score(y_test, y_pred, average = "weighted"),
                         roc_auc_score(y_test, final_model.predict_proba(x_test), multi_class = "ovr")]})

,metric,score
0,accuracy,0.920502
1,balanced_accuracy,0.927651
2,precision,0.921733
3,recall,0.920502
4,f1,0.920552
5,roc,0.992428


In [147]:
print(classification_report(y_test, 
                      final_model.predict(x_test),
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        28
           1       0.91      0.96      0.93        50
           2       0.89      0.89      0.89        37
           3       0.89      0.90      0.89        80
           4       0.97      0.89      0.93        44

    accuracy                           0.92       239
   macro avg       0.93      0.93      0.93       239
weighted avg       0.92      0.92      0.92       239



In [106]:
y_pred = final_model.predict(x_test)
pd.DataFrame({"Cluster" : ["SAID", "SIDD", "SIRD", "MOD", "MARD"],
              "precision" : precision_score(y_test, y_pred, average = None),
              "recall" : recall_score(y_test, y_pred, average = None),
              "f1" : f1_score(y_test, y_pred, average = None),
              "roc" : roc_auc_score(y_test, final_model.predict_proba(x_test), average = None, multi_class = "ovr")})

,Cluster,precision,recall,f1,roc
0,SAID,1.000000,1.000000,1.000000,1.000000
1,SIDD,0.905660,0.960000,0.932039,0.991323
2,SIRD,0.891892,0.891892,0.891892,0.993444
3,MOD,0.888889,0.900000,0.894410,0.982154
4,MARD,0.975000,0.886364,0.928571,0.995221


# Export model

In [44]:
import pickle

In [46]:
from joblib import dump, load